# RL-Chess-Engine — Experiment C (both levers together)

Experiment A (10× compute) moved Elo vs random from −21 to −7. Experiment B (self-play
material weight 0.30 at baseline compute) moved it to +28. **Experiment C runs both at
once** to test whether the levers *compound* or *substitute*.

Runs **only** the combined arm and compares it to the published A/B figures, so it costs
about one scaled stage (roughly 5–8 GPU-hours), not a full rerun. Writes
`docs/experiment-c-results.md` and commits it back. The shipped model checkpoint is
**not** touched.

**Before running (Kaggle):**
- Settings → Accelerator → **GPU T4 x2**
- Settings → Internet → **On**
- Add-ons → Secrets → `GITHUB_TOKEN` (a token with `repo` scope), attached to this notebook
- Then **Save Version → Save & Run All (Commit)** so it runs in the background (up to 12 h).

## 1 — Confirm GPU

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), 'No GPU detected: set Accelerator to GPU T4 x2.'
print('CUDA OK:', torch.cuda.get_device_name(0))

## 2 — Clone the repo

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['GH_TOKEN'] = UserSecretsClient().get_secret('GITHUB_TOKEN')
!git clone https://$GH_TOKEN@github.com/Danny-397/RL-Chess-Engine.git
%cd RL-Chess-Engine
!git config user.email 'dlichtenberger91@gmail.com'
!git config user.name  'Danny-397'
!pip install -q 'python-chess>=1.11'

## 3 — Smoke test (CPU, ~3 min, numbers meaningless)

In [ ]:
!python run_experiment.py --quick --only-combined
!rm -f experiment_out/combined.result.json   # so the real run below is not skipped

## 4 — Experiment C on GPU
Resumable: if the session drops, re-run from step 2; a finished stage is skipped.

In [ ]:
!python run_experiment.py --device cuda --only-combined
!cat docs/experiment-c-results.md

## 5 — Commit the write-up and charts (not the checkpoint)

In [ ]:
import subprocess
!git add docs/experiment-c-results.md
!git add assets/term_combined.png assets/progress_combined.png 2>/dev/null || true
!git diff --cached --name-only
!git commit -m 'Experiment C: scaled compute with the material assist'
push = subprocess.run(['git', 'push', 'origin', 'HEAD:main'], capture_output=True, text=True)
print(push.stdout, push.stderr)
if push.returncode != 0:
    raise RuntimeError('Push failed: the results are only in this session. Download '
                       'docs/experiment-c-results.md from the file browser before it ends.')
print('Pushed.')